In [ ]:
# 03 Experiment A: reference regression baseline
#
# Question. How well does ResNet50 (RadImageNet initialization) trained with plain MSE predict the 0-4 quality score on the frozen validation partition? A is the reference for B and C and is B's lambda = 0 row.
# Hypothesis. None to test: A defines the baseline.
# Configuration. resnet50_regression (sigmoid head), MSE on scores normalized to [0, 1], Adam lr 1e-3, batch 64, 30 epochs, seed 42, bn_eps 1.001e-5, centre crop 224 + 2x-1, selection by validation PLCC. Spec: docs/experiments/a-reference.md.
# Expected outputs (in the run directory): run_manifest.json, config.snapshot.json, history.json, checkpoints/best.pt, metrics_validation.json, predictions_validation.csv.
# Cell labels. [SAFE] inspection only | [TRAINS] trains, OFF by default | [READS RUN] reads a completed run.
# No cell in this notebook reads the official test set.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ct_iqa.config.loader import load_study_config
from ct_iqa.data.splits import load_manifest
from ct_iqa.experiment import report
from ct_iqa.experiment.grid import apply_overrides
from ct_iqa.experiment.run import run_experiment
from ct_iqa.experiment.runs import read_predictions, run_summary, runs_table, select_run
from ct_iqa.models.weights import sha256_file


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# 1. Configuration [SAFE]
# Loads and validates the experiment configuration. Nothing runs.
CONFIG = "configs/experiments/a_reference.yaml"
cfg = load_study_config(CONFIG)        # strict: unknown keys or invalid values raise here, before anything runs
show(f"Configuration: configs/base.yaml + {CONFIG}", report.config_frame(**{cfg.run.experiment_id: cfg.to_dict()}))


In [ ]:
# 2. Preflight [SAFE]
# Checks inputs WITHOUT training and WITHOUT building or running a model: frozen split manifest (hash, disjoint partitions, coverage), pretrained weights SHA-256, image files present (counted, not read), CUDA availability, existing runs. Expected output: a table where every row is True.
labels = json.loads(Path(cfg.data.train_json_path).read_text())
manifest = load_manifest(cfg.data.split_manifest, labels)      # raises if the frozen manifest was altered
sizes = {p: len(manifest[p]) for p in ("train", "validation", "calibration")}
weights = Path(cfg.model.pretrained_weights)
n_images = len(list(Path(cfg.data.train_image_dir).glob("*.tif*")))
preflight = {
    "split manifest verified (hash, disjoint, complete)": True,
    "partitions are 800 / 100 / 100": list(sizes.values()) == [800, 100, 100],
    "pretrained weights file present": weights.is_file(),
    "pretrained weights SHA-256 matches config": weights.is_file() and sha256_file(weights) == cfg.model.pretrained_sha256,
    "1000 training images present (counted, not read)": n_images == 1000,
}
show("Preflight checks", pd.DataFrame({"passed": preflight}))
show("Environment", pd.DataFrame({"value": {
    "split manifest SHA-256": manifest["manifest_sha256"], "split seed": manifest["seed"],
    "torch": torch.__version__, "CUDA available": torch.cuda.is_available(),
    "CUDA device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}}))
print("PREFLIGHT PASSED" if all(preflight.values()) else "PREFLIGHT FAILED - fix the False rows before training")


In [ ]:
# 3. Train [TRAINS - OFF by default]
# Expensive: one full training run of the configured epochs per seed, through the shared engine (ct_iqa.experiment.run.run_experiment). Each run gets a NEW run directory and is never overwritten. Progress: one log line per epoch (* = new selected checkpoint) and the run directory when it starts. On an error the run is marked failed in its run_manifest.json and the error is raised here.
# DEVICE is used exactly as written: if it is cuda and CUDA is unavailable the cell stops instead of silently using the CPU. SEEDS defaults to the configured seed; a multi-seed design is a protocol decision that has not been approved yet.
RUN_TRAINING = False
DEVICE = "cuda"
SEEDS = [cfg.train.seed]

if RUN_TRAINING:
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(f"DEVICE={DEVICE!r} but CUDA is not available in this kernel; select the GPU kernel or set DEVICE = 'cpu' explicitly")
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
    for seed in SEEDS:
        overrides = {"train": {"device": DEVICE, "seed": seed}}
        new_run = run_experiment(apply_overrides(cfg, overrides), config_path=CONFIG, overrides=overrides)
        print(f"COMPLETED: {new_run.as_posix()}  (RUN_ID = {new_run.name!r})")
else:
    print("RUN_TRAINING is False: nothing was trained.")


In [ ]:
# 4. Runs of this experiment [SAFE]
# Every run directory with its status. failed and started (interrupted) runs are listed so that nothing is hidden, but they are never used for results below.
rows = runs_table(cfg.run.experiment_id)
show(f"Runs of {cfg.run.experiment_id}", pd.DataFrame(rows).set_index("run_id").dropna(axis=1, how="all") if rows else pd.DataFrame({"status": {"(none)": "no run yet: PENDING"}}))


In [ ]:
# 5. Select and verify one completed run [READS RUN]
# Set RUN_ID to a run directory name from the table above. With exactly one completed run it may stay None; with several the cell stops rather than guess. Verification: status completed, every artifact matches its recorded SHA-256, history complete, same frozen split and bn_eps as the current config, official test set unused. All later cells are skipped while no run is selected.
RUN_ID = None

run_dir = select_run(cfg.run.experiment_id, RUN_ID)
if run_dir is None:
    print(f"No completed run of {cfg.run.experiment_id}: results are PENDING.")
else:
    S = run_summary(run_dir)                               # raises on failed / incomplete / altered runs
    m, history, metrics = S["manifest"], S["history"], S["metrics"]
    verified = {
        "status is completed": m["status"] == "completed",
        "artifact SHA-256 all match": True,                # run_summary raised otherwise
        "history marked complete": history.get("complete", True),
        "split manifest SHA-256 = current frozen manifest": m["split_manifest_sha256"] == manifest["manifest_sha256"],
        "bn_eps = current config": m["bn_eps"] == cfg.model.bn_eps,
        "official test set unused": m["official_test_set_used"] is False,
    }
    show(f"Verification of {run_dir.as_posix()}", pd.DataFrame({"passed": verified}))
    if not all(verified.values()):
        raise RuntimeError("this run does not match the current protocol; do not interpret it")


In [ ]:
# 6. Selected checkpoint [READS RUN]
# The checkpoint is the epoch with the highest validation PLCC (raw 0-4 scale), evaluated after every epoch; a later epoch replaces it only on a strict improvement. The validation partition therefore SELECTS the reported model: validation metrics below are optimistic (selection-biased).
if run_dir:
    best = next(e for e in history["epochs"] if e["epoch"] == history["best_epoch"])
    show("Checkpoint selection record", pd.DataFrame({"value": {
        "selection policy": history["selection_policy"], "selected epoch (1-based)": history["best_epoch"],
        "of epochs": len(history["epochs"]), "selected validation PLCC": round(history["best_value"], 4),
        "validation PLCC at the final epoch": round(history["epochs"][-1]["val_plcc"], 4),
        "checkpoint file": (run_dir / "checkpoints" / "best.pt").as_posix(),
        "checkpoint SHA-256": m["artifact_sha256"]["checkpoints/best.pt"]}}))


In [ ]:
# 7. Training curves [READS RUN]
# Left: training and validation objective per epoch (normalized 0-1 score space). Right: validation PLCC and SROCC. The dashed line marks the selected epoch.
if run_dir:
    report.plot_history(history, f"{m['run_id']}  (n_train = 800, n_val = 100)")
    plt.show()
    show("Per-epoch history (selected = True: new best checkpoint at that epoch)", report.history_frame(history).round(4))


In [ ]:
# 8. Validation metrics at the selected checkpoint [READS RUN]
# Raw 0-4 score scale, n = 100 validation images. SELECTION-BIASED (the same images chose the epoch); not comparable with historical official-test numbers, which come from a different partition and protocol.
if run_dir:
    show(f"Validation metrics, epoch {metrics['epoch_1based']}, n = {metrics['n']}, {metrics['scale']} (selection-biased)",
         report.metrics_frame(metrics))


In [ ]:
# 9. Prediction and residual diagnostics [READS RUN]
# Scatter against the identity line, residual distribution, residual vs label, and errors per label range with the number of images in each range (ranges with few images are noisy).
if run_dir:
    p = read_predictions(run_dir)
    y, f = np.array(p["label_0_4"]), np.array(p["prediction_0_4"])
    report.plot_prediction_scatter(y, f, f"Validation: prediction vs label (n = {len(y)})")
    plt.show()
    report.plot_residuals(y, f, "Validation residuals")
    plt.show()
    show("Error by label range (score units)", report.error_by_score_bin(y, f).round(3))


In [ ]:
# 10. Reproducibility record [READS RUN]
# Everything needed to identify and reproduce this run; all paths are relative to the repository root. The full configuration is in config.snapshot.json inside the run directory.
if run_dir:
    show("Reproducibility record", report.reproducibility_frame(S))

# 11. Limitations and next step
# Limitations. Validation metrics are selection-biased; one seed cannot separate the model from training noise; 100 validation images give wide uncertainty on correlations; the split is image-level (no patient identifiers), so validation images may share patients with training images; the 2x-1 input convention for the RadImageNet weights is a documented assumption.
# Next step. Record the completed RUN_ID; it is the reference run that notebook 04 needs for B's selection rule and that notebook 06 needs for the comparison. The final comparison partition and the seed design are pending a protocol decision (docs/experiment-protocol.md).
